# Fetch Teams and Standings

**Database: MySQL** (`mysql-connector-python`)

Before running:
1. MySQL **Server** must be installed and running (MySQL Workbench is only the client).
2. Put your real MySQL root password in the **Connection settings** cell.

In [ ]:
%pip install requests pandas mysql-connector-python

In [1]:
import os
import re
import time
import requests
import pandas as pd
import mysql.connector
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from IPython.display import display

STANDINGS_URL = "https://api-web.nhle.com/v1/standings/now"
REQUEST_DELAY = 0.5

## Connection settings

Use the same host, port, user and password you use to open your connection in MySQL Workbench
(check the connection's settings with the wrench icon; the password is the one you chose during MySQL install).

In [2]:
DB_HOST     = os.getenv("DB_HOST", "127.0.0.1")
DB_PORT     = int(os.getenv("DB_PORT", "3306"))
DB_USER     = os.getenv("DB_USER", "root")
DB_PASSWORD = os.getenv("DB_PASSWORD", "root")   # <-- change to YOUR MySQL password
DB_NAME     = os.getenv("DB_NAME", "NHL_Analytics_Hub")

def get_connection(database=None):
    return mysql.connector.connect(
        host=DB_HOST, port=DB_PORT, user=DB_USER, password=DB_PASSWORD,
        database=database, connection_timeout=15,
    )

# Connection test with readable diagnosis
try:
    test = get_connection()
    print("Connected OK. MySQL server version:", test.get_server_info())
    test.close()
except mysql.connector.Error as exc:
    print("Could not connect to MySQL.")
    print("Error:", exc, "\n")
    if exc.errno == 2003:
        print("-> Server not reachable. Start the MySQL service (services.msc -> MySQL80, or run "
              "'net start MySQL80' in an Admin terminal), and check DB_HOST / DB_PORT.")
    elif exc.errno == 1045:
        print("-> Wrong username or password. Set DB_PASSWORD to the password you use in Workbench.")
    elif exc.errno == 1251 or "caching_sha2" in str(exc):
        print("-> Authentication plugin problem. Update the connector: pip install -U mysql-connector-python")
    raise

Connected OK. MySQL server version: 8.0.46


C:\Users\bhuva\AppData\Local\Temp\ipykernel_25864\3928290088.py:16: DeprecationWarning: Call to deprecated function get_server_info. Reason: 
    The property counterpart 'server_info' should be used instead.

  print("Connected OK. MySQL server version:", test.get_server_info())


## Fetch Standings

In [3]:
def fetch_standings():
    retry = Retry(total=3, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504],
                  allowed_methods=["GET"], respect_retry_after_header=True)
    with requests.Session() as session:
        session.mount("https://", HTTPAdapter(max_retries=retry))
        try:
            response = session.get(STANDINGS_URL, timeout=(10, 30))
            response.raise_for_status()
            payload = response.json()
        finally:
            time.sleep(REQUEST_DELAY)
    if not isinstance(payload, dict) or not isinstance(payload.get("standings"), list) or not payload["standings"]:
        raise ValueError("API response must contain a non-empty standings list; database unchanged.")
    return payload

data = fetch_standings()
print("Top-level keys:", list(data))
print("Team rows:", len(data["standings"]))
display(data["standings"][0])

Top-level keys: ['wildCardIndicator', 'standingsDateTimeUtc', 'standings']
Team rows: 32


{'conferenceAbbrev': 'E',
 'conferenceHomeSequence': 8,
 'conferenceL10Sequence': 1,
 'conferenceName': 'Eastern',
 'conferenceRoadSequence': 1,
 'conferenceSequence': 1,
 'date': '2026-10-30',
 'divisionAbbrev': 'M',
 'divisionHomeSequence': 5,
 'divisionL10Sequence': 1,
 'divisionName': 'Metropolitan',
 'divisionRoadSequence': 1,
 'divisionSequence': 1,
 'gameTypeId': 2,
 'gamesPlayed': 15,
 'goalDifferential': 20,
 'goalDifferentialPctg': 1.333333,
 'goalAgainst': 36,
 'goalFor': 56,
 'goalsForPctg': 3.733333,
 'homeGamesPlayed': 6,
 'homeGoalDifferential': 5,
 'homeGoalsAgainst': 13,
 'homeGoalsFor': 18,
 'homeLosses': 1,
 'homeOtLosses': 1,
 'homePoints': 9,
 'homeRegulationPlusOtWins': 4,
 'homeRegulationWins': 4,
 'homeTies': 0,
 'homeWins': 4,
 'l10GamesPlayed': 10,
 'l10GoalDifferential': 16,
 'l10GoalsAgainst': 22,
 'l10GoalsFor': 38,
 'l10Losses': 1,
 'l10OtLosses': 1,
 'l10Points': 17,
 'l10RegulationPlusOtWins': 8,
 'l10RegulationWins': 8,
 'l10Ties': 0,
 'l10Wins': 8,
 'l

## Prepare Teams and Standings

In [4]:
TEAM_FIELDS = ["team_abbrev", "team_name", "conference_name", "division_name", "logo_url"]
STAT_MAP = {"games_played": "gamesPlayed", "wins": "wins", "losses": "losses",
            "ot_losses": "otLosses", "points": "points", "goals_for": "goalFor",
            "goals_against": "goalAgainst", "home_wins": "homeWins",
            "away_wins": "roadWins", "streak_count": "streakCount"}
STAT_FIELDS = list(STAT_MAP) + ["streak_type"]

def text_value(value, limit, required=False):
    if isinstance(value, dict):
        value = value.get("default")
    if value is None:
        if required:
            raise ValueError("Missing required text")
        return None
    if not isinstance(value, str) or len(value) > limit or (required and not value.strip()):
        raise ValueError(f"Invalid text: {value!r}")
    return value.strip()

def integer_value(value):
    if value is None:
        return None
    if isinstance(value, bool) or not re.fullmatch(r"[0-9]+", str(value)):
        raise ValueError(f"Invalid nonnegative integer: {value!r}")
    result = int(value)
    if result > 2147483647:
        raise ValueError("Integer exceeds SQL INT")
    return result

def parse_records(payload):
    unique = {}
    for raw in payload["standings"]:
        if not isinstance(raw, dict):
            raise ValueError("Standing record must be an object")
        team = dict(zip(TEAM_FIELDS, [
            text_value(raw.get("teamAbbrev"), 10, True),
            text_value(raw.get("teamName"), 100, True),
            text_value(raw.get("conferenceName"), 50),
            text_value(raw.get("divisionName"), 50),
            text_value(raw.get("teamLogo"), 4000)]))
        team["team_abbrev"] = team["team_abbrev"].upper()
        season = str(raw.get("seasonId", ""))
        if not re.fullmatch(r"[0-9]{8}", season) or int(season[4:]) != int(season[:4]) + 1:
            raise ValueError(f"Invalid seasonId: {season!r}")
        stats = {name: integer_value(raw.get(api)) for name, api in STAT_MAP.items()}
        stats["streak_type"] = text_value(raw.get("streakCode"), 20)
        record = {**team, "season": season, **stats}
        key = (team["team_abbrev"], season)
        if key in unique and unique[key] != record:
            raise ValueError(f"Conflicting duplicate: {key}")
        unique[key] = record
    records = list(unique.values())
    if len({r["season"] for r in records}) != 1:
        raise ValueError("Expected one season from standings/now")
    return records

records = parse_records(data)
season = records[0]["season"]
teams_df = pd.DataFrame(records)[TEAM_FIELDS]
standings_df = pd.DataFrame(records)[["team_abbrev", "season"] + STAT_FIELDS]
print("Season:", season, "Validated rows:", len(records))
display(teams_df.head(), standings_df.head())

Season: 20262027 Validated rows: 32


,team_abbrev,team_name,conference_name,division_name,logo_url
0,CAR,Carolina Hurricanes,Eastern,Metropolitan,https://assets.nhle.com/logos/nhl/svg/CAR_ligh...
1,MIN,Minnesota Wild,Western,Central,https://assets.nhle.com/logos/nhl/svg/MIN_ligh...
2,EDM,Edmonton Oilers,Western,Pacific,https://assets.nhle.com/logos/nhl/svg/EDM_ligh...
3,BOS,Boston Bruins,Eastern,Atlantic,https://assets.nhle.com/logos/nhl/svg/BOS_ligh...
4,VGK,Vegas Golden Knights,Western,Pacific,https://assets.nhle.com/logos/nhl/svg/VGK_ligh...


,team_abbrev,season,games_played,wins,losses,ot_losses,points,goals_for,goals_against,home_wins,away_wins,streak_count,streak_type
0,CAR,20262027,15,11,2,2,24,56,36,4,7,3,W
1,MIN,20262027,15,9,3,3,21,47,35,4,5,1,OT
2,EDM,20262027,13,9,2,2,20,56,43,4,5,1,W
3,BOS,20262027,14,9,4,1,19,50,32,5,4,2,L
4,VGK,20262027,15,9,5,1,19,53,44,6,3,1,L


## Create Tables

In [5]:
TEAMS_SQL = """
CREATE TABLE IF NOT EXISTS teams (
    team_id INT AUTO_INCREMENT PRIMARY KEY,
    team_abbrev VARCHAR(10) NOT NULL UNIQUE,
    team_name VARCHAR(100) NOT NULL,
    conference_name VARCHAR(50),
    division_name VARCHAR(50),
    logo_url TEXT
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
"""

STANDINGS_SQL = """
CREATE TABLE IF NOT EXISTS standings (
    standing_id INT AUTO_INCREMENT PRIMARY KEY,
    team_id INT NOT NULL,
    season VARCHAR(20) NOT NULL,
    games_played INT, wins INT, losses INT, ot_losses INT,
    points INT, goals_for INT, goals_against INT,
    home_wins INT, away_wins INT, streak_type VARCHAR(20), streak_count INT,
    UNIQUE (team_id, season),
    FOREIGN KEY (team_id) REFERENCES teams(team_id)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
"""

## Save to MySQL

Hint: Existing records are updated when you run this cell again.

In [6]:
conn = get_connection()

try:
    cur = conn.cursor()
    cur.execute(f"CREATE DATABASE IF NOT EXISTS `{DB_NAME}` CHARACTER SET utf8mb4")
    cur.execute(f"USE `{DB_NAME}`")
    cur.execute(TEAMS_SQL)
    cur.execute(STANDINGS_SQL)

    # DDL commits implicitly in MySQL; start a clean transaction for the data load.
    conn.commit()
    conn.start_transaction()

    for team in records:
        cur.execute("""
            INSERT INTO teams
                (team_abbrev, team_name, conference_name, division_name, logo_url)
            VALUES (%s, %s, %s, %s, %s)
            ON DUPLICATE KEY UPDATE
                team_name=%s, conference_name=%s, division_name=%s, logo_url=%s
        """, (team["team_abbrev"], team["team_name"], team["conference_name"],
              team["division_name"], team["logo_url"], team["team_name"],
              team["conference_name"], team["division_name"], team["logo_url"]))

    cur.execute("SELECT team_abbrev, team_id FROM teams")
    team_id_lookup = {row[0]: row[1] for row in cur.fetchall()}

    for team in records:
        team_id = team_id_lookup[team["team_abbrev"]]
        values = (
            team["games_played"], team["wins"], team["losses"],
            team["ot_losses"], team["points"], team["goals_for"],
            team["goals_against"], team["home_wins"], team["away_wins"],
            team["streak_type"], team["streak_count"]
        )

        cur.execute("""
            INSERT INTO standings
                (team_id, season, games_played, wins, losses, ot_losses,
                 points, goals_for, goals_against, home_wins, away_wins,
                 streak_type, streak_count)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            ON DUPLICATE KEY UPDATE
                games_played=%s, wins=%s, losses=%s, ot_losses=%s,
                points=%s, goals_for=%s, goals_against=%s,
                home_wins=%s, away_wins=%s, streak_type=%s, streak_count=%s
        """, (team_id, team["season"]) + values + values)

        cur.execute("""
            SELECT games_played, wins, losses, ot_losses, points,
                   goals_for, goals_against, home_wins, away_wins,
                   streak_type, streak_count
            FROM standings WHERE team_id=%s AND season=%s
        """, (team_id, team["season"]))
        saved_rows = cur.fetchall()
        if len(saved_rows) != 1 or tuple(saved_rows[0]) != values:
            raise ValueError("Standings check failed: " + team["team_abbrev"])

    cur.execute("""
        SELECT t.team_abbrev, t.team_name, t.conference_name,
               s.season, s.points, s.wins, s.losses, s.ot_losses
        FROM teams AS t
        JOIN standings AS s ON t.team_id = s.team_id
        WHERE s.season=%s
        ORDER BY s.points DESC, t.team_abbrev
        LIMIT 10
    """, (season,))

    columns = [column[0] for column in cur.description]
    check_df = pd.DataFrame(cur.fetchall(), columns=columns)
    conn.commit()
    print("Teams and standings saved successfully. Season:", season)

except Exception:
    conn.rollback()
    raise

finally:
    conn.close()

display(check_df)

Teams and standings saved successfully. Season: 20262027


,team_abbrev,team_name,conference_name,season,points,wins,losses,ot_losses
0,CAR,Carolina Hurricanes,Eastern,20262027,24,11,2,2
1,MIN,Minnesota Wild,Western,20262027,21,9,3,3
2,EDM,Edmonton Oilers,Western,20262027,20,9,2,2
3,BOS,Boston Bruins,Eastern,20262027,19,9,4,1
4,VGK,Vegas Golden Knights,Western,20262027,19,9,5,1
5,FLA,Florida Panthers,Eastern,20262027,18,8,4,2
6,MTL,Montréal Canadiens,Eastern,20262027,18,8,3,2
7,NJD,New Jersey Devils,Eastern,20262027,18,9,5,0
8,UTA,Utah Mammoth,Western,20262027,18,9,5,0
9,OTT,Ottawa Senators,Eastern,20262027,17,8,5,1
